# SynBioCrow 2.3 — Normalize Galaxy Literature Pathways\nConverts all 77 Galaxy-SynBioCAD literature pathways into the SynBioCrow literature benchmark schema. Preserves original InChI, EC/enzyme annotations, Galaxy step numbers, and external matching scores. A deterministic 12-pathway development split is selected independently of prediction performance.\n

In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_galaxy_normalize")
INGEST=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_dataset2_ingest")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1200,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

raw=INGEST/"galaxy_dataset2_raw.json"
assert raw.is_file(),f"Missing ingest artifact: {raw}"

run([sys.executable,str(REPO/"scripts/normalize_galaxy_literature_2_3.py"),
     "--raw-json",str(raw),
     "--output-dir",str(OUT),
     "--development-count","12"],timeout=600)

norm=json.loads((OUT/"galaxy_literature_benchmark_normalized.json").read_text())
print("\nDEVELOPMENT PATHWAYS")
for r in norm["records"]:
    if r["split"]=="development":
        print(r["pathway_id"],"|",r["target_name"],"|",r["chassis"]["organism"],
              "| steps",len(r["reactions"]),"| sources",r["chassis"]["source_metabolites"])

bundle=OUT/"SynBioCrow_2_3_GALAXY_LITERATURE_NORMALIZED_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")):
        z.write(p,p.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
